In [2]:
// Parameters
let _batch_mode: &str = "true"; // jamais lu (le carnet Rust n'a pas de traducteur papermill)

# Complexity-03c — Naviguer le Zoo exécutable (Rust)

**Navigation** : [README de la série](README.md) · [version Python — source du portage](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) · [jumeau C#](Complexity-03c-ComplexityZoo-Navigation-CSharp.ipynb) · [03 — Plus de temps, plus de problèmes](Complexity-03-HierarchieTemporelle-Python.ipynb) · [04 — Décider sans connaître la suite](Complexity-04-OnlineAlgorithms-Python.ipynb)

Ce carnet est le **jumeau Rust** du carnet Python homonyme. Il refait les mêmes expériences avec la
bibliothèque standard de Rust uniquement (aucune crate externe, aucun `:dep`) : les algorithmes sont
réécrits, pas appelés depuis une bibliothèque. Contrairement au jumeau C#, le port Rust **reprend le
flux pseudo-aléatoire de CPython** (générateur MT19937 réimplémenté depuis les mots, même graine,
même ordre des tirages) : les instances mesurées et les valeurs numériques imprimées sont donc
identiques à la version Python, dont les sorties committées servent de témoin de conformité. Chaque
cellule étant autonome sous evcxr, elle rejoue les tirages des cellules précédentes pour se replacer
au même point du flux global.

Inclusions vérifiées sur des instances réelles, oracles jouets séparant deux mondes, carte des non-séparations.
Le [Complexity Zoo](https://complexityzoo.net) — registre fondé par Scott Aaronson — comme terrain de navigation.

## 0. Pourquoi ce notebook

Approfondissement du palier [03 — Hiérarchie en temps](Complexity-03-HierarchieTemporelle-Python.ipynb),
compagnon de [03b — Hartmanis & Stearns](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb) : après le
temps compté et sa hiérarchie stricte, ce carnet prend la carte entière des classes et la fait naviguer.
La série se lit par ailleurs
[01 — Compter des pas](Complexity-01-StepCounting-Python.ipynb),
[02 — Vérifier contre trouver](Complexity-02-P-NP-Reduction-Python.ipynb),
[04 — Décider sans connaître la suite](Complexity-04-OnlineAlgorithms-Python.ipynb),
[04b — Conjectures online](Complexity-04b-OnlineConjectures-Secretary-KServer-Python.ipynb),
[05b — Aaronson & Arkhipov](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb) et
[06 — Déquantification](Complexity-06b-AaronsonGottesman-Dequantification-Python.ipynb).

Le Zoo recense plus de 500 classes de complexité. Une table d'inclusions recopiée serait une étagère ;
ce notebook **navigue** le Zoo : chaque arête d'inclusion est **tournee sur des instances** (solveurs reels,
temps et memoire mesures), la question des séparations est rendue **concrete par des oracles jouets**
(Baker–Gill–Solovay, deux mondes construits et executes), et la carte des non-séparations se termine sur
la barriere qui les protege : la relativisation.

Famille Aaronson (Epic #16781, veine 2) : le Zoo et la Zoo Map sont l'objet pédagogique nommé par le brief —
ce carnet en est l'organe d'entrée.

In [3]:
// Toutes les experiences de ce carnet sont deterministes : chaque generateur
// d'instances est execute sous graine fixee, les chronos sont encadres.
// Le generateur rejoue le flux MT19937 de CPython (init_by_array) : les
// temoins de conformite sont les sorties commitees du carnet Python source.
struct Mt19937 { mt: [u32; 624], idx: usize, words: u64 }   // words : diagnostic (controle de conformite)

impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {                      // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624, words: 0 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        self.words += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits, DEUX mots
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 { // 1 <= k <= 32
        (self.next_u32() >> (32 - k)) as u64
    }
    fn randbelow(&mut self, n: u64) -> u64 {   // rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randrange(&mut self, stop: u64) -> u64 { self.randbelow(stop) }          // randrange(n)
    fn randrange_de_a(&mut self, debut: i64, fin: i64) -> i64 {                  // randrange(1, n+1)
        debut + self.randbelow((fin - debut) as u64) as i64
    }
}
fn chrono<T, F: Fn() -> T>(f: F) -> (T, f64) {
    // Minimum de passes encadrees, comme la source ; rend le resultat de la derniere.
    // Ne consomme jamais d'aleatoire : l'instance est generee hors chrono.
    let mut duree_min = f64::INFINITY;
    let mut dernier = None;
    for _ in 0..3 {
        let debut = std::time::Instant::now();
        dernier = Some(f());
        let duree = debut.elapsed().as_secs_f64();
        if duree < duree_min {
            duree_min = duree;
        }
    }
    (dernier.unwrap(), duree_min)
}
let _graine = Mt19937::new(20260925); // graine du carnet entier, identique a la source

println!("Chronometre et generateur severes en place.");

Chronometre et generateur severes en place.


## 1. La chaîne des inclusions, tournée sur instances

La ligne du Zoo la plus citée — $L \subseteq NL \subseteq P \subseteq NP \subseteq PSPACE$ — est
d'ordinaire une affirmation. Ici, **chaque arête porte un témoin exécutable** :

- $L \subseteq NL$ : l'atteignabilité $s \leadsto t$ décidée par un **vérificateur certificat lu une fois**,
  dont la mémoire de travail hors certificat tient en deux compteurs $O(\log n)$ — contre le DFS déterministe
  dont la pile grandit avec le graphe ;
- $NL \subseteq P$ : le même problème décidé par parcours déterministe, temps mesuré ;
- $P \subseteq NP$ : le certificat du 3-SAT **vérifié** en temps polynomial — vérifier n'est pas trouver ;
  la comparaison qui suit oppose 2-SAT (implications + composantes fortement connexes, temps linéaire)
  à 3-SAT (DPLL, exponentiel) — sur **deux familles distinctes** : 2-SAT à $n = 50 \dots 800$ avec $3n$ clauses,
  3-SAT à $n = 10 \dots 30$ avec $4{,}6n$ clauses.

Chaque inclusion est tournée sur instance : la même question d'atteignabilité est décidée deux fois, par des machines dont seule la mémoire diffère — le contraste mesuré EST l'inclusion.


In [4]:
// --- 1a. Atteignabilite : le DFS deterministe (P) et sa pile ---
struct Mt19937 { mt: [u32; 624], idx: usize, words: u64 }   // words : diagnostic (controle de conformite)

impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {                      // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624, words: 0 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        self.words += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits, DEUX mots
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 { // 1 <= k <= 32
        (self.next_u32() >> (32 - k)) as u64
    }
    fn randbelow(&mut self, n: u64) -> u64 {   // rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randrange(&mut self, stop: u64) -> u64 { self.randbelow(stop) }          // randrange(n)
    fn randrange_de_a(&mut self, debut: i64, fin: i64) -> i64 {                  // randrange(1, n+1)
        debut + self.randbelow((fin - debut) as u64) as i64
    }
}
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}
fn chrono<T, F: Fn() -> T>(f: F) -> (T, f64) {
    // Minimum de passes encadrees, comme la source ; rend le resultat de la derniere.
    // Ne consomme jamais d'aleatoire : l'instance est generee hors chrono.
    let mut duree_min = f64::INFINITY;
    let mut dernier = None;
    for _ in 0..3 {
        let debut = std::time::Instant::now();
        dernier = Some(f());
        let duree = debut.elapsed().as_secs_f64();
        if duree < duree_min {
            duree_min = duree;
        }
    }
    (dernier.unwrap(), duree_min)
}
fn dfs_atteignable(adj: &[Vec<usize>], s: usize, t: usize) -> (bool, usize) {
    let mut vu = vec![false; adj.len()];
    vu[s] = true;
    let mut pile = vec![s];
    let mut pile_max = 1usize; // la pile contient deja s
    while let Some(u) = pile.pop() {
        if u == t {
            return (true, pile_max);
        }
        for &v in &adj[u] {
            if !vu[v] {
                vu[v] = true;
                pile.push(v);
                if pile.len() > pile_max {
                    pile_max = pile.len(); // maj au PUSH, pas au pop
                }
            }
        }
    }
    (false, pile_max)
}
fn graphe_aleatoire(rng: &mut Mt19937, n: usize, densite: f64) -> Vec<Vec<usize>> {
    // int(densite) : troncature comme Python -- densite 4 donne 4 arcs, 2.5 donne 2.
    let arcs = densite as usize;
    let mut adj = vec![Vec::new(); n];
    for u in 0..n {
        for _ in 0..arcs {
            let v = rng.randrange(n as u64) as usize;
            adj[u].push(v); // boucles et doublons possibles, comme en Python
        }
    }
    adj
}
let mut rng = Mt19937::new(20260925);
for n in [40usize, 100, 250, 600] {
    let adj = graphe_aleatoire(&mut rng, n, 4.0);
    let ((ok, pile_max), duree) = chrono(|| dfs_atteignable(&adj, 0, n - 1));
    println!("n={:>4}  atteint={}  pile max (espace explore)={:>4}  temps={:>8.3} ms",
             n, py_bool(ok), pile_max, duree * 1e3);
} ;

n=  40  atteint=True  pile max (espace explore)=  16  temps=   0.001 ms


n= 100  atteint=True  pile max (espace explore)=  35  temps=   0.001 ms


n= 250  atteint=True  pile max (espace explore)= 103  temps=   0.002 ms


n= 600  atteint=True  pile max (espace explore)= 190  temps=   0.002 ms


### Lecture du résultat — le DFS paie son espace

La pile du DFS atteint une fraction de $n$ : l'algorithme déterministe naturel consomme un espace **lineaire**.
C'est le témoin du contraste qui suit : la même question d'atteignabilité est le problème **NL-complet** de
référence — une machine nondéterministe n'a besoin, elle, que de **deviner** le chemin sommet par sommet.

La sortie donne la pile mesurée sur la famille croissante — 16, 35, 103 puis 190 sommets empilés, sur la
famille à **4 arcs par sommet** — pendant que le vérificateur à deux compteurs, lui, ne garde que son
certificat : ce coût est celui de **parcourir** l'espace au lieu de le **certifier**, et la paire de traces
est la première inclusion rendue visible — toute la différence entre résoudre et vérifier qui structure
le reste du Zoo.


In [5]:
// --- 1b. Le verificateur NL : certificat lu UNE fois, deux compteurs en memoire ---
struct Mt19937 { mt: [u32; 624], idx: usize, words: u64 }   // words : diagnostic (controle de conformite)

impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {                      // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624, words: 0 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        self.words += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits, DEUX mots
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 { // 1 <= k <= 32
        (self.next_u32() >> (32 - k)) as u64
    }
    fn randbelow(&mut self, n: u64) -> u64 {   // rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randrange(&mut self, stop: u64) -> u64 { self.randbelow(stop) }          // randrange(n)
    fn randrange_de_a(&mut self, debut: i64, fin: i64) -> i64 {                  // randrange(1, n+1)
        debut + self.randbelow((fin - debut) as u64) as i64
    }
}
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}
fn chrono<T, F: Fn() -> T>(f: F) -> (T, f64) {
    // Minimum de passes encadrees, comme la source ; rend le resultat de la derniere.
    // Ne consomme jamais d'aleatoire : l'instance est generee hors chrono.
    let mut duree_min = f64::INFINITY;
    let mut dernier = None;
    for _ in 0..3 {
        let debut = std::time::Instant::now();
        dernier = Some(f());
        let duree = debut.elapsed().as_secs_f64();
        if duree < duree_min {
            duree_min = duree;
        }
    }
    (dernier.unwrap(), duree_min)
}
fn dfs_atteignable(adj: &[Vec<usize>], s: usize, t: usize) -> (bool, usize) {
    let mut vu = vec![false; adj.len()];
    vu[s] = true;
    let mut pile = vec![s];
    let mut pile_max = 1usize; // la pile contient deja s
    while let Some(u) = pile.pop() {
        if u == t {
            return (true, pile_max);
        }
        for &v in &adj[u] {
            if !vu[v] {
                vu[v] = true;
                pile.push(v);
                if pile.len() > pile_max {
                    pile_max = pile.len(); // maj au PUSH, pas au pop
                }
            }
        }
    }
    (false, pile_max)
}
fn graphe_aleatoire(rng: &mut Mt19937, n: usize, densite: f64) -> Vec<Vec<usize>> {
    // int(densite) : troncature comme Python -- densite 4 donne 4 arcs, 2.5 donne 2.
    let arcs = densite as usize;
    let mut adj = vec![Vec::new(); n];
    for u in 0..n {
        for _ in 0..arcs {
            let v = rng.randrange(n as u64) as usize;
            adj[u].push(v); // boucles et doublons possibles, comme en Python
        }
    }
    adj
}
fn verificateur_nl(adj: &[Vec<usize>], s: usize, t: usize, certificat: &[usize]) -> (bool, usize) {
    // Memoire de travail hors certificat : u (sommet courant) et pas (compteur).
    // Le certificat est un flux parcouru une seule fois, jamais relu.
    let n = adj.len();
    let mut u = s;
    let mut pas = 0usize;
    for &v in certificat {
        pas += 1;
        if pas > n + 1 {
            return (false, pas);
        }
        if !adj[u].contains(&v) {
            return (false, pas);
        }
        u = v;
        if u == t {
            return (true, pas);
        }
    }
    (u == t, pas)
}

fn chemin_temoins(adj: &[Vec<usize>], s: usize, t: usize) -> Option<Vec<usize>> {
    // Le "prouveur" (hors machine) : un chemin quelconque s->t, par BFS.
    let mut file = std::collections::VecDeque::new();
    file.push_back(s);
    let mut pere: Vec<Option<usize>> = vec![None; adj.len()];
    let mut vus = vec![false; adj.len()];
    vus[s] = true;
    while let Some(u) = file.pop_front() {
        if u == t {
            let mut chemin = Vec::new();
            let mut courant = Some(u);
            while let Some(x) = courant {
                chemin.push(x);
                courant = pere[x];
            }
            chemin.reverse();
            return Some(chemin);
        }
        for &v in &adj[u] {
            if !vus[v] {
                vus[v] = true;
                pere[v] = Some(u);
                file.push_back(v);
            }
        }
    }
    None
}

let mut rng = Mt19937::new(20260925);
// Le carnet Python n'a qu'UN flux global : rejeu de la consommation de la cellule
// precedente (quatre graphes a 4 arcs, resultats jetes) avant les tirages propres.
for n in [40usize, 100, 250, 600] {
    let _ = graphe_aleatoire(&mut rng, n, 4.0);
}
println!("{:>5} {:>8} {:>9} {:>5} {:>15}", "n", "atteint", "verif NL", "pas", "memoire (mots)");
let mut accord_total = true;
// Motif du jumeau C# : retenir le dernier couple (graphe, chemin) qui a un chemin.
let mut dernier_graphe: Option<Vec<Vec<usize>>> = None;
let mut dernier_chemin: Option<Vec<usize>> = None;
let mut dernier_n = 0usize;
for n in [40usize, 100, 250, 600] {
    let adj = graphe_aleatoire(&mut rng, n, 2.5); // int(2.5) = 2 arcs par sommet
    let (ok_dfs, _) = dfs_atteignable(&adj, 0, n - 1);
    let temoin = chemin_temoins(&adj, 0, n - 1);
    let (ok_nl, pas) = match &temoin {
        Some(chemin) => verificateur_nl(&adj, 0, n - 1, &chemin[1..]),
        None => (false, n + 2),
    };
    let accord = ok_dfs == ok_nl;
    accord_total &= accord;
    println!("{:>5} {:>8} {:>9} {:>5} {:>15}   accord={}",
             n, py_bool(ok_dfs), py_bool(ok_nl), pas, 2, py_bool(accord));
    if temoin.is_some() {
        dernier_graphe = Some(adj);
        dernier_chemin = temoin;
        dernier_n = n;
    }
}
println!("Accord verificateur/DFS sur toute la famille : {}", py_bool(accord_total));

// Un faux temoin : le chemin du dernier graphe atteignable, ampute de son arrivee.
// Le verificateur doit le refuser -- c'est la seconde moitie de "exactement".
let adj = dernier_graphe.unwrap();
let chemin = dernier_chemin.unwrap();
let faux: Vec<usize> = chemin[1..chemin.len() - 1].to_vec();
let (ok_faux, _) = verificateur_nl(&adj, 0, dernier_n - 1, &faux);
println!("Faux temoin (arrivee amputee) refuse par le verificateur : {}", py_bool(!ok_faux));

    n  atteint  verif NL   pas  memoire (mots)


   40     True      True     4               2   accord=True


  100     True      True     9               2   accord=True


  250     True      True     5               2   accord=True


  600     True      True     9               2   accord=True


Accord verificateur/DFS sur toute la famille : True


Faux temoin (arrivee amputee) refuse par le verificateur : True


### Lecture du résultat — deux mots contre une pile

Le vérificateur accepte les paires atteignables (accord sur les quatre tailles) et **refuse le témoin amputé**
de son arrivée. Sa mémoire de travail — deux mots, indépendants de $n$ — opposée à la pile du DFS, qui croît
avec $n$ (16, 35, 103 puis 190 sommets empilés sur la famille à **4 arcs par sommet** mesurée en 1a ; le
tableau du vérificateur porte une autre famille, de **densité nominale 2,5** — le générateur arrondit par
`int()`, donc **deux arcs par sommet** en pratique), **est** l'arête $L \subseteq NL$
rendue visible : même question, mais une mémoire que le DFS paie linéairement là où le vérificateur la paie
en deux compteurs — au prix d'un certificat deviné (le nondéterminisme est simulé ici par le prouveur BFS
hors machine ; chaque tentative de la machine ne consomme toujours que deux compteurs).

In [6]:
// --- 1c. P contre NP : 2-SAT lineaire, 3-SAT exponentiel, deux familles distinctes ---
use std::collections::HashMap;
struct Mt19937 { mt: [u32; 624], idx: usize, words: u64 }   // words : diagnostic (controle de conformite)

impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(seed), 0 <= seed < 2**32
    fn new_by_array(key: &[u32]) -> Mt19937 {                      // init_by_array(key) de CPython
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624, words: 0 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        self.words += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits, DEUX mots
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn getrandbits(&mut self, k: u32) -> u64 { // 1 <= k <= 32
        (self.next_u32() >> (32 - k)) as u64
    }
    fn randbelow(&mut self, n: u64) -> u64 {   // rejet sur getrandbits(n.bit_length())
        let k = 64 - n.leading_zeros();
        let mut r = self.getrandbits(k);
        while r >= n { r = self.getrandbits(k); }
        r
    }
    fn randrange(&mut self, stop: u64) -> u64 { self.randbelow(stop) }          // randrange(n)
    fn randrange_de_a(&mut self, debut: i64, fin: i64) -> i64 {                  // randrange(1, n+1)
        debut + self.randbelow((fin - debut) as u64) as i64
    }
}
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}
fn chrono<T, F: Fn() -> T>(f: F) -> (T, f64) {
    // Minimum de passes encadrees, comme la source ; rend le resultat de la derniere.
    // Ne consomme jamais d'aleatoire : l'instance est generee hors chrono.
    let mut duree_min = f64::INFINITY;
    let mut dernier = None;
    for _ in 0..3 {
        let debut = std::time::Instant::now();
        dernier = Some(f());
        let duree = debut.elapsed().as_secs_f64();
        if duree < duree_min {
            duree_min = duree;
        }
    }
    (dernier.unwrap(), duree_min)
}
fn dfs_atteignable(adj: &[Vec<usize>], s: usize, t: usize) -> (bool, usize) {
    let mut vu = vec![false; adj.len()];
    vu[s] = true;
    let mut pile = vec![s];
    let mut pile_max = 1usize; // la pile contient deja s
    while let Some(u) = pile.pop() {
        if u == t {
            return (true, pile_max);
        }
        for &v in &adj[u] {
            if !vu[v] {
                vu[v] = true;
                pile.push(v);
                if pile.len() > pile_max {
                    pile_max = pile.len(); // maj au PUSH, pas au pop
                }
            }
        }
    }
    (false, pile_max)
}
fn graphe_aleatoire(rng: &mut Mt19937, n: usize, densite: f64) -> Vec<Vec<usize>> {
    // int(densite) : troncature comme Python -- densite 4 donne 4 arcs, 2.5 donne 2.
    let arcs = densite as usize;
    let mut adj = vec![Vec::new(); n];
    for u in 0..n {
        for _ in 0..arcs {
            let v = rng.randrange(n as u64) as usize;
            adj[u].push(v); // boucles et doublons possibles, comme en Python
        }
    }
    adj
}
fn deux_sat(clauses: &[(i64, i64)], nb_var: usize) -> bool {
    // Kosaraju (recursif, n petit : jusqu'a 2n = 1600 sommets) sur le graphe
    // d'implications : temps lineaire, donc 2-SAT dans P.
    let n = nb_var;
    let lit = |l: i64| -> usize {
        // litteral signe -> index interne : x_i -> i-1, non-x_i -> i-1+n
        let base = (l.abs() - 1) as usize;
        if l > 0 { base } else { base + n }
    };
    let neg = |i: usize| -> usize { if i < n { i + n } else { i - n } };
    let mut adj = vec![Vec::new(); 2 * n];
    let mut radj = vec![Vec::new(); 2 * n];
    for &(a, b) in clauses {
        let (la, lb) = (lit(a), lit(b));
        adj[neg(la)].push(lb); // (a ou b) = (non-a -> b) et (non-b -> a)
        adj[neg(lb)].push(la);
        radj[lb].push(neg(la));
        radj[la].push(neg(lb));
    }
    let mut vu = vec![false; 2 * n];
    let mut ordre = Vec::new();
    fn dfs1(adj: &[Vec<usize>], vu: &mut [bool], ordre: &mut Vec<usize>, u: usize) {
        vu[u] = true;
        for &v in &adj[u] {
            if !vu[v] {
                dfs1(adj, vu, ordre, v);
            }
        }
        ordre.push(u);
    }
    for s in 0..2 * n {
        if !vu[s] {
            dfs1(&adj, &mut vu, &mut ordre, s);
        }
    }
    let mut comp = vec![-1i64; 2 * n];
    fn dfs2(radj: &[Vec<usize>], comp: &mut [i64], u: usize, c: i64) {
        comp[u] = c;
        for &v in &radj[u] {
            if comp[v] == -1 {
                dfs2(radj, comp, v, c);
            }
        }
    }
    let mut c = 0i64;
    for &s in ordre.iter().rev() {
        if comp[s] == -1 {
            dfs2(&radj, &mut comp, s, c);
            c += 1;
        }
    }
    // La table val n'est jamais imprimee : seul le verdict sert ici.
    !(0..n).any(|i| comp[i] == comp[i + n])
}

fn trois_sat_dpll(clauses: &[Vec<i64>], nb_var: usize)
    -> (bool, Option<HashMap<usize, bool>>, usize) {
    // DPLL (propagation de unite + branchement) avec compteur de noeuds explores :
    // metrique deterministe -- le temps varie d'une machine a l'autre, pas le nombre
    // de noeuds, propriete de l'instance ET de la strategie de branchement.
    fn resoudre(partiel: &HashMap<usize, bool>, restantes: &[Vec<i64>], noeuds: &mut usize)
        -> Option<HashMap<usize, bool>> {
        *noeuds += 1; // compteur a l'ENTREE du noeud, rejets compris
        if restantes.iter().any(|c| c.is_empty()) {
            return None; // clause vide verifiee a l'entree, avant propagation
        }
        let mut partiel = partiel.clone();
        let mut restantes: Vec<Vec<i64>> = restantes.to_vec();
        loop {
            // litteral unitaire : premier c[0] de la premiere clause de longueur 1, dans l'ordre
            let l = match restantes.iter().find(|c| c.len() == 1) {
                Some(c) => c[0],
                None => break,
            };
            let mut nouvelles = Vec::new();
            for c in &restantes {
                if c.contains(&l) {
                    continue;
                }
                if c.contains(&-l) {
                    let c2: Vec<i64> = c.iter().copied().filter(|&x| x != -l).collect();
                    if c2.is_empty() {
                        return None;
                    }
                    nouvelles.push(c2);
                } else {
                    nouvelles.push(c.clone());
                }
            }
            restantes = nouvelles;
            partiel.insert(l.abs() as usize, l > 0);
        }
        if restantes.is_empty() {
            return Some(partiel);
        }
        let l = restantes[0][0]; // literal de branchement : premier de la premiere clause
        for &choix in &[l, -l] {
            // cand filtre "choix not in c" puis retire "-choix" (peut laisser une
            // clause vide, rattrapee au noeud fils)
            let cand: Vec<Vec<i64>> = restantes.iter()
                .filter(|c| !c.contains(&choix))
                .map(|c| c.iter().copied().filter(|&x| x != -choix).collect())
                .collect();
            let mut partiel2 = partiel.clone();
            partiel2.insert(choix.abs() as usize, choix > 0);
            if let Some(resultat) = resoudre(&partiel2, &cand, noeuds) {
                return Some(resultat);
            }
        }
        None
    }
    let mut noeuds = 0usize;
    let solution = resoudre(&HashMap::new(), clauses, &mut noeuds);
    (solution.is_some(), solution, noeuds)
}

fn verifier_3sat(clauses: &[Vec<i64>], solution: &HashMap<usize, bool>) -> bool {
    // L'arete P inclus dans NP : verifier un certificat est polynomial.
    clauses.iter().all(|c| {
        c.iter().any(|&x| (x > 0) == *solution.get(&(x.abs() as usize)).unwrap_or(&false))
    })
}

fn instance_2sat(rng: &mut Mt19937, n: usize, m: usize) -> Vec<(i64, i64)> {
    // Ordre des tirages : par litteral, randrange(1, n+1) PUIS random() du signe,
    // a puis b -- jamais entrelace autrement.
    let mut clauses = Vec::new();
    for _ in 0..m {
        let a = rng.randrange_de_a(1, n as i64 + 1) * (if rng.random() < 0.5 { 1 } else { -1 });
        let b = rng.randrange_de_a(1, n as i64 + 1) * (if rng.random() < 0.5 { 1 } else { -1 });
        clauses.push((a, b));
    }
    clauses
}

fn instance_3sat_plantee(rng: &mut Mt19937, n: usize, m: usize) -> Vec<Vec<i64>> {
    // 3-CNF aleatoire (litteral 50/50). Densite 4.6n : juste au-dessus du seuil de
    // satisfiabilite 4.267n, la ou les instances aleatoires sont les plus dures pour DPLL.
    let mut clauses = Vec::new();
    for _ in 0..m {
        let mut c: Vec<i64> = Vec::new();
        while c.len() < 3 {
            let v = rng.randrange_de_a(1, n as i64 + 1);
            // un tirage rejete ne consomme PAS de random() : le signe n'est tire qu'a l'acceptation
            if !c.iter().any(|x| x.abs() == v) {
                c.push(if rng.random() < 0.5 { v } else { -v });
            }
        }
        clauses.push(c);
    }
    clauses
}

fn instance_satisfiable(rng: &mut Mt19937, n: usize, m: usize, essais_max: usize) -> Vec<Vec<i64>> {
    let mut cl = Vec::new();
    for _ in 0..essais_max {
        cl = instance_3sat_plantee(rng, n, m);
        let (sat, _, _) = trois_sat_dpll(&cl, n);
        if sat {
            return cl;
        }
    }
    cl // rend la DERNIERE instance meme si echec, comme la source
}

let mut rng = Mt19937::new(20260925);
// Rejeu du flux global : la cellule 1a a consomme quatre graphes a 4 arcs, la
// cellule 1b quatre graphes a 2 arcs -- resultats jetes, seuls les tirages comptent.
for n in [40usize, 100, 250, 600] {
    let _ = graphe_aleatoire(&mut rng, n, 4.0);
}
for n in [40usize, 100, 250, 600] {
    let _ = graphe_aleatoire(&mut rng, n, 2.5);
}

// 2-SAT : le probleme de P -- de 50 a 800 variables (seize fois plus), le temps suit de pres
println!("2-SAT (Kosaraju) -- moyenne sur 5 instances :");
for n in [50usize, 200, 800] {
    let mut cumul = 0.0f64;
    for _ in 0..5 {
        let clauses2 = instance_2sat(&mut rng, n, 3 * n);
        let (_, d2) = chrono(|| deux_sat(&clauses2, n));
        cumul += d2;
    }
    println!("  n={:>4}   {:>8.3} ms", n, cumul / 5.0 * 1e3);
}

// 3-SAT : le probleme NP-complet -- meme question existentielle, densite au seuil
println!("3-SAT (DPLL, densite 4.6n) -- moyenne sur 20 instances :");
let tailles = [10usize, 15, 20, 25, 30];
let mut noeuds_moyens = [0.0f64; 5];
let mut temps_moyens = [0.0f64; 5];
for (k, &n) in tailles.iter().enumerate() {
    // m = int(4.6 * n) : 46/69/92/114/138 -- 114 pour n = 25 (IEEE : 4.6*25 = 114.99...)
    let m = (4.6 * n as f64) as usize;
    let mut total_noeuds = 0usize;
    let mut cumul = 0.0f64;
    for _ in 0..20 {
        let clauses3 = instance_3sat_plantee(&mut rng, n, m);
        let ((_, _, nds), d3) = chrono(|| trois_sat_dpll(&clauses3, n));
        total_noeuds += nds;
        cumul += d3;
    }
    noeuds_moyens[k] = total_noeuds as f64 / 20.0;
    temps_moyens[k] = cumul / 20.0;
    println!("  n={:>3}   noeuds DPLL : {:>8.1}   {:>8.3} ms",
             n, total_noeuds as f64 / 20.0, temps_moyens[k] * 1e3);
}

let facteur_noeuds = (noeuds_moyens[4] / noeuds_moyens[0]).powf(1.0 / 20.0);
let facteur_temps = (temps_moyens[4] / temps_moyens[0]).powf(1.0 / 20.0);
println!("Facteur de croissance par variable ajoutee (noeuds explores, deterministe) : ~{:.2}x", facteur_noeuds);
println!("Facteur de croissance par variable ajoutee (temps) : ~{:.2}x", facteur_temps);

// L'arete P inclus dans NP, dans l'autre sens : le certificat se verifie en polynomial
let mut certificats = 0usize;
for _ in 0..20 {
    let clauses3 = instance_satisfiable(&mut rng, 12, 4 * 12, 12);
    let ((sat3, sol3, _), _) = chrono(|| trois_sat_dpll(&clauses3, 12));
    if sat3 {
        if let Some(sol) = sol3 {
            if verifier_3sat(&clauses3, &sol) {
                certificats += 1;
            }
        }
    }
}
println!("Certificats 3-SAT verifies en temps polynomial par verifier_3sat : {} / 20", certificats);

2-SAT (Kosaraju) -- moyenne sur 5 instances :


  n=  50      0.016 ms


  n= 200      0.071 ms


  n= 800      0.399 ms


3-SAT (DPLL, densite 4.6n) -- moyenne sur 20 instances :


  n= 10   noeuds DPLL :     10.3      0.057 ms


... (3 outputs left out)

  n= 30   noeuds DPLL :     63.9      1.822 ms


Facteur de croissance par variable ajoutee (noeuds explores, deterministe) : ~1.10x


Facteur de croissance par variable ajoutee (temps) : ~1.19x


Certificats 3-SAT verifies en temps polynomial par verifier_3sat : 20 / 20


### Lecture du résultat — la même question existentielle, deux pentes

2-SAT reste sous quelques millisecondes même à $n = 800$ : composantes fortement connexes en temps linéaire —
un problème de $P$ : de $n = 50$ à $n = 800$ soit **seize fois** plus de variables, et le temps passe de
quelques dixièmes de milliseconde à quelques millisecondes : il suit de près $n$, comme le veut le temps
linéaire. 3-SAT, sur des instances **bien plus petites** ($n = 10 \dots 30$ contre $50 \dots 800$) et tirées à la
densité $4{,}6n$ — juste au-dessus du seuil de satisfiabilité $4{,}267n$, là où la recherche est la plus dure — paie
déjà des millisecondes à $n = 30$ : chaque variable ajoutée **multiplie** le coût (facteur de temps mesuré
ci-dessus, de l'ordre de $1{,}1$ à $1{,}2$ par variable) au lieu de l'ajouter — et le coût par noeud grossit encore,
car le nombre de clauses suit $n$. La pente est modeste à l'échelle jouet, mais elle compose : l'exercice 3
l'extrapole et franchit l'heure
de calcul pendant que 2-SAT, lui, aura juste suivi $n$. L'arête $P \subseteq NP$ a son témoin dans l'autre
sens : `verifier_3sat` contrôle chaque certificat en un parcours polynomial, quand même le trouver résiste.

## 2. Deux mondes d'oracles — Baker, Gill & Solovay exécutés

Le Zoo dit ce qui est **prouvé** (inclusions) et ce qui est **ouvert** ($P$ vs $NP$ en tête). La question
naturelle : pourquoi pas une diagonalisation, comme pour la hiérarchie en temps du [03](Complexity-03-HierarchieTemporelle-Python.ipynb) ?
La réponse de Baker–Gill–Solovay (1975) est un **exécutable** : il existe un oracle $O$ tel que $P^O \ne NP^O$,
et un oracle $B$ tel que $P^B = NP^B$. Deux mondes cohérents — donc aucune technique qui « se relativise »
(diagonalisation incluse) ne peut séparer $P$ de $NP$ dans le nôtre. Ce paragraphe **construit et exécute les deux**.

Échelle jouet assumée : chaque machine $P$ simulée émet au plus $n$ requêtes d'oracle sur une entrée de
taille $n$ (stand-in borné du polynôme), la machine $NP$ a droit à l'énumération exhaustive de $2^n$ témoins.
La structure de la preuve — stades, longueurs fraîches, basculement unique — est celle du texte original.

In [7]:
// --- 2a. Monde A : oracle diagonal ou P^O != NP^O ---
use std::collections::HashMap;

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

fn machine_p(oracle: &HashMap<String, i64>, x: &str, defaut: i64, budget_marges: usize) -> i64 {
    // Machine deterministe : parcourt les temoins y de |y| = |x| dans l'ordre
    // lexicographique et s'arrete apres n + budget_marges requetes (stand-in du
    // polynome). Repond par vote majoritaire, ou par son biais 'defaut' si elle
    // n'a rien vu (elle est arbitraire).
    let n = x.len();
    let limite = n + budget_marges;
    let mut vus = 0usize;
    let mut positifs = 0usize;
    for entier_y in 0..(1usize << n) {
        let y = format!("{:0width$b}", entier_y, width = n);
        let requete = format!("{}#{}", x, y);
        if *oracle.get(&requete).unwrap_or(&0) != 0 {
            positifs += 1;
        }
        vus += 1;
        if vus >= limite {
            break;
        }
    }
    if positifs == 0 {
        return defaut;
    }
    if positifs > limite / 2 { 1 } else { 0 }
}

fn langage_np(oracle: &HashMap<String, i64>, x: &str) -> i64 {
    // Cote NP : enumeration exhaustive des temoins (stand-in exponentiel autorise).
    let n = x.len();
    for entier_y in 0..(1usize << n) {
        let y = format!("{:0width$b}", entier_y, width = n);
        if *oracle.get(&format!("{}#{}", x, y)).unwrap_or(&0) != 0 {
            return 1;
        }
    }
    0
}

fn construire_oracle_separant(nb_machines: usize) -> (HashMap<String, i64>, Vec<(usize, usize, i64, i64)>) {
    let mut oracle = HashMap::new();
    let mut stades = Vec::new();
    for i in 1..=nb_machines {
        // Longueur fraiche : assez grande pour que la machine i (n requetes sur 2^n
        // paires) ne puisse pas tout voir, distincte des longueurs precedentes.
        // Biais alterne : les machines paires repondent 0 par defaut, les impaires 1.
        let n_i = 6 + 2 * i;
        let x_i = format!("{:0width$b}", i, width = n_i);
        let reponse_machine = machine_p(&oracle, &x_i, (i % 2) as i64, 0);
        if reponse_machine == 0 {
            // elle dit non : on plante UN temoin -> L_O(x_i) = 1, elle a tort.
            let y_etoile = format!("{:0width$b}", (i * 7919) % (1usize << n_i), width = n_i);
            oracle.insert(format!("{}#{}", x_i, y_etoile), 1);
            stades.push((i, n_i, reponse_machine, 1));
        } else {
            // elle dit oui sans preuve : on ne plante rien -> L_O(x_i) = 0, elle a tort.
            stades.push((i, n_i, reponse_machine, 0));
        }
    }
    (oracle, stades)
}

let (oracle_a, stades) = construire_oracle_separant(4);
println!("Stades de la diagonalisation (machine, taille, reponse machine, valeur reelle) :");
for &(i, n_i, rep, reel) in &stades {
    let verdict = if rep != reel { "machine vaincue" } else { "ERREUR" };
    println!("  M_{} : n={:>2}  M_i^O(x_i)={}  L_O(x_i)={}  -> {}", i, n_i, rep, reel, verdict);
}
let separation = stades.iter().all(|&(_, _, rep, reel)| rep != reel);
// Verification croisee : le cote NP (exhaustif) lit-il bien L_O ?
let accord_np = stades.iter().all(|&(i, _, _, reel)| {
    let x_i = format!("{:0width$b}", i, width = 6 + 2 * i);
    langage_np(&oracle_a, &x_i) == reel
});
println!("Toutes les machines P vaincues : {}", py_bool(separation));
println!("Le temoin exhaustif (cote NP) confirme L_O sur chaque stade : {}", py_bool(accord_np));

Stades de la diagonalisation (machine, taille, reponse machine, valeur reelle) :


  M_1 : n= 8  M_i^O(x_i)=1  L_O(x_i)=0  -> machine vaincue


  M_2 : n=10  M_i^O(x_i)=0  L_O(x_i)=1  -> machine vaincue


  M_3 : n=12  M_i^O(x_i)=1  L_O(x_i)=0  -> machine vaincue


  M_4 : n=14  M_i^O(x_i)=0  L_O(x_i)=1  -> machine vaincue


Toutes les machines P vaincues : True


Le temoin exhaustif (cote NP) confirme L_O sur chaque stade : True


### Lecture du résultat — pourquoi la machine perd à chaque stade

Chaque machine $M_i$ ne voit que $n_i$ des $2^{n_i}$ paires $(x_i, y)$ : le basculement d'**un seul** témoin
lui échappe forcément. C'est la mécanique du texte original — stades successifs, longueurs fraîches, un
basculement par machine — rendue exécutable : dans ce monde, $P^O \ne NP^O$, **prouvé par construction**.
L'échelle est jouet, la structure est celle de 1975.

La construction appelle sa symétrie : le monde B qui suit applique le même geste d'oracle dans l'autre sens — un oracle assez complet pour qu'une requête suffise — et obtient l'égalité. Deux mondes, deux réponses, mêmes axiomes : c'est cette coexistence qui arme la barrière de la section suivante.


In [8]:
// --- 2b. Monde B : oracle complet ou P^B = NP^B ---
fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

fn oracle_qbf<F: Fn(&str, &str) -> bool>(h: F, x: &str) -> i64 {
    // L'oracle (stand-in PSPACE) evalue la formule quantifiee "existe y, h(x, y)"
    // par enumeration exhaustive. C'est SON travail : la machine ne paie rien.
    let n = x.len();
    for entier_y in 0..(1usize << n) {
        if h(x, &format!("{:0width$b}", entier_y, width = n)) {
            return 1;
        }
    }
    0
}

fn machine_p_avec_oracle_qbf<F: Fn(&str, &str) -> bool>(x: &str, h: F) -> (i64, i64) {
    // La machine P emet UNE requete et recopie la reponse : le langage existentiel
    // "existe y, h(x, y)" est lui-meme un QBF a un quantificateur -- une seule
    // question a l'oracle.
    (oracle_qbf(h, x), 1)
}

fn machine_np_sans_oracle<F: Fn(&str, &str) -> bool>(x: &str, h: F) -> i64 {
    // Le meme langage vu du cote NP SANS oracle : l'enumeration est payee par la machine.
    let n = x.len();
    for entier_y in 0..(1usize << n) {
        if h(x, &format!("{:0width$b}", entier_y, width = n)) {
            return 1;
        }
    }
    0
}

fn h_jouet(x: &str, y: &str) -> bool {
    x.matches('1').count() + y.matches('1').count() >= x.len()
}

let echantillon: Vec<String> = [0usize, 5, 13, 21, 31].iter().map(|&k| format!("{:05b}", k)).collect();
let mut accord_b = true;
for x in &echantillon {
    let (rep_p, requetes) = machine_p_avec_oracle_qbf(x, h_jouet);
    let rep_np = machine_np_sans_oracle(x, h_jouet);
    accord_b &= rep_p == rep_np;
    println!("x={}  P^B(x)={} ({} requete)  NP^B(x)={}  accord={}",
             x, rep_p, requetes, rep_np, py_bool(rep_p == rep_np));
}
println!("Monde B : la machine P a une requete egale la machine NP exhaustive : {}", py_bool(accord_b));

x=00000  P^B(x)=1 (1 requete)  NP^B(x)=1  accord=True


x=00101  P^B(x)=1 (1 requete)  NP^B(x)=1  accord=True


x=01101  P^B(x)=1 (1 requete)  NP^B(x)=1  accord=True


x=10101  P^B(x)=1 (1 requete)  NP^B(x)=1  accord=True


x=11111  P^B(x)=1 (1 requete)  NP^B(x)=1  accord=True


Monde B : la machine P a une requete egale la machine NP exhaustive : True


### Lecture du résultat — la barrière de relativisation, vue du jouet

Monde A : $P^O \ne NP^O$, par construction. Monde B : $P^B = NP^B$ — le langage existentiel
« $\exists y, h(x,y)$ » est **lui-même** une formule quantifiée, et une machine $P$ armée de l'oracle
le décide en une requête. Les deux mondes sont cohérents avec les mêmes axiomes : **aucune méthode qui
se contente de simuler des machines relativisées** — la diagonalisation en premier — ne peut donc trancher
$P$ vs $NP$. C'est la raison pour laquelle la carte du Zoo a un centre ouvert, et pourquoi il a fallu des
techniques non relativisantes (arithmétisation, IP = PSPACE) pour les séparations qui ont été prouvées.

## 3. La carte du Zoo — ce qui est mesuré, ce qui est cité, ce qui est ouvert

Les trois niveaux épistémiques de la série ([04b](Complexity-04b-OnlineConjectures-Secretary-KServer-Python.ipynb))
appliqués à la carte : **Mesuré** (témoin exécuté dans ce carnet ou dans un sibling), **Cité** (théorème
établi, non rejoué ici), **Ouvert** (non séparé — et chaque arête ouverte a sa raison propre : la barrière de
relativisation, l'absence de certificateur général, ou une technique non relativisante).

La table qui suit assemble le relevé depuis les témoins des sections 1 et 2 : chaque arête de la chaîne reçoit son statut et son témoin — exécuté dans ce carnet, établi ailleurs, ou ouvert avec la barrière qui l'explique.


In [9]:
// --- 3. La carte assemblee depuis les temoins du carnet et de la serie ---
let carte: [(&str, &str, &str); 8] = [
    // (arete, statut, temoin)
    ("L incl NL", "Mesure", "verificateur a 2 compteurs contre pile DFS (section 1)"),
    ("NL incl P", "Mesure", "le meme probleme decide par parcours deterministe (section 1)"),
    ("P incl NP", "Mesure", "certificat 3-SAT verifie en temps polynomial (section 1)"),
    ("NP incl PSPACE", "Cite", "un certificat polynomial s'evalue en espace polynomial (tire du socle 01)"),
    ("P != EXPTIME", "Cite", "theoreme de hierarchie en temps -- rejoue au toy-scale dans 03/03b"),
    ("P vs NP", "Ouvert", "mondes d'oracles contradictoires (section 2) : la diagonalisation ne tranche pas"),
    ("NP vs coNP", "Ouvert", "equivaut a : aucun certificateur general pour la non-satisfiabilite"),
    ("NP vs PSPACE", "Ouvert", "IP = PSPACE (arithmetisation, non relativisante) n'eclaire pas cette arete"),
];
println!("{:<16} {:<8} {}", "Arete", "Statut", "Temoin");
println!("{}", "-".repeat(88));
for (arete, statut, temoin) in carte {
    println!("{:<16} {:<8} {}", arete, statut, temoin);
}
let completes = carte.iter().filter(|c| c.1 == "Mesure").count();
println!("\nArêtes à témoin exécutable dans ce carnet : {} / {}", completes, carte.len());

Arete            Statut   Temoin


----------------------------------------------------------------------------------------


L incl NL        Mesure   verificateur a 2 compteurs contre pile DFS (section 1)


NL incl P        Mesure   le meme probleme decide par parcours deterministe (section 1)


P incl NP        Mesure   certificat 3-SAT verifie en temps polynomial (section 1)


NP incl PSPACE   Cite     un certificat polynomial s'evalue en espace polynomial (tire du socle 01)


P != EXPTIME     Cite     theoreme de hierarchie en temps -- rejoue au toy-scale dans 03/03b


P vs NP          Ouvert   mondes d'oracles contradictoires (section 2) : la diagonalisation ne tranche pas


NP vs coNP       Ouvert   equivaut a : aucun certificateur general pour la non-satisfiabilite


NP vs PSPACE     Ouvert   IP = PSPACE (arithmetisation, non relativisante) n'eclaire pas cette arete


Arêtes à témoin exécutable dans ce carnet : 3 / 8


### Lecture du résultat — trois arêtes exécutables sur huit

```mermaid
flowchart LR
    L[L] --> NL[NL] --> P[P] --> NP[NP] --> PS[PSPACE]
    P --> EX[EXPTIME]
    P -.open.- NP
    NP -.open.- coNP[coNP]
    NP -.open.- PS
```

En trait plein : inclusions établies (celles de la section 1 portent leur témoin exécutable ;
l'arête $P \to EXPTIME$ est la stricte $P \subsetneq EXPTIME$ de la hiérarchie en temps,
[03](Complexity-03-HierarchieTemporelle-Python.ipynb)).
En pointillés : les arêtes ouvertes — **trois**, et chacune a sa raison propre : la diagonalisation
ne tranche pas $P$ vs $NP$ (section 2), un certificateur général pour la non-satisfiabilité
manquerait pour $NP$ vs $coNP$, et l'arithmétisation — non relativisante — n'éclaire pas
$NP$ vs $PSPACE$.

La carte assemblée sépare ce que le carnet **mesure** (L⊆NL, NL⊆P et P⊆NP : les trois arêtes de la section 1, chacune avec son témoin exécuté), ce qu'il **cite** (NP⊆PSPACE, P≠EXPTIME — théorèmes établis, non rejoués ici) et ce qui reste **ouvert** (P vs NP, NP vs coNP, NP vs PSPACE — la section 2 n'a montré que pour la première
pourquoi les oracles n'y suffisent pas ; les deux autres ont leur raison propre). Le bilan de la dernière ligne, 3/8 arêtes à témoin exécutable, est la signature du carnet : la hiérarchie du Zoo n'est pas une tour de certitudes, c'est un relevé différencié où chaque arête porte son statut épistémique — et c'est ce statut, pas la vérité, que l'exécution mesure.


## 4. Encart — le gap Mathlib du Zoo

Le même geste que [03b](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb), [05b](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb) et [06](Complexity-06b-AaronsonGottesman-Dequantification-Python.ipynb) :
confronter la formalisation du carnet à l'état de [Mathlib](https://github.com/leanprover-community/mathlib4).

- **Ce qui existe** : une couche `Computability/` réelle — machines de Turing (`TuringMachine`),
  fonctions partielles récursives, décidabilité et indécidabilité (`Primrec`, `Partrec`, le problème de
  l'arrêt), réductions many-one. C'est le socle $L$ de la chaîne, du côté prouvé.
- **Ce qui n'existe pas (encore)** : les classes à bornage — pas de $\mathrm{TIME}(f)$ paramétrée, pas de $NP$,
  pas de $NL$, pas de machine à oracle, pas de théorème de hiérarchie, pas de Baker–Gill–Solovay.
  Le Zoo est absent : Mathlib formalise la calculabilité, pas (encore) la complexité.
- **Grade** : les arêtes de la section 1 sont **exécutables** (mesuré dans ce carnet), pas prouvées ;
  leur formalisation est un programme de recherche ouvert, pas un gap d'encodage. Le jour où Aaronson
  entre en Lean dans ce dépôt, ce sera par la veine 5 de l'Epic (#16781) — un lake IIT/ICT dédié.

## 5. Exercices

### Exercice 1 — 2-SAT côté NL

L'atteignabilité est NL-complète, et 2-SAT se réduit à l'atteignabilité dans le graphe d'implications.
Écrire le vérificateur certificat-lu-une-fois qui décide 2-SAT : le certificat est la suite des sommets
d'un chemin dans le graphe d'implications witnessant (ou réfutant) la satisfiabilité, la mémoire de
travail reste bornée en $O(\log n)$ mots.

Les trois exercices reprennent un fil chacun : le vérificateur NL (§1a-1b), l'oracle diagonal (§2a), la pente DPLL (§1c) — le carnet a exécuté les cas témoins, l'exercice demande d'étendre la même mécanique d'un cran.


In [10]:
// Exercice 1 : a completer
fn verificateur_2sat_nl(nb_var: usize, clauses: &[(i64, i64)], certificat: &[i64]) -> Option<bool> {
    // TODO etudiant : decider 2-SAT en lisant le certificat une seule fois,
    // memoire de travail bornee (sommet courant + compteur), comme verificateur_nl.
    // Le certificat : suite de litteraux (chemin dans le graphe d'implications).
    // retour : Some(true) / Some(false)
    None // TODO etudiant
}
// Indice : non-a -> b veut dire que si le chemin atteint non-a, il doit passer par b.
let _exercice_1 = verificateur_2sat_nl;

### Exercice 2 — étendre le monde A d'un cran

Une cinquième machine $M_5$ arrive après coup. Écrire `etendre_oracle` : un nouveau stade qui choisit
une longueur fraîche, simule $M_5$ sur l'oracle courant, et bascule exactement un témoin pour la vaincre —
sans défaire les stades 1 à 4 (les réponses déjà acquises doivent rester fausses pour leur machine).

In [11]:
// Exercice 2 : a completer
fn etendre_oracle(
    oracle: &std::collections::HashMap<String, i64>,
    stades: &[(usize, usize, i64, i64)],
    indice_nouvelle_machine: usize,
) -> (Option<std::collections::HashMap<String, i64>>, Option<Vec<(usize, usize, i64, i64)>>) {
    // TODO etudiant : ajouter le stade de la machine indice_nouvelle_machine
    // (longueur fraiche n = 6 + 2*indice, simulation, basculement d'UN temoin au plus).
    // retour : (oracle_etendu, stades_etendus)
    (None, None) // TODO etudiant
}
// Indice : la longueur doit etre nouvelle -- sinon une machine precedente pourrait revoir son temoin.
let _exercice_2 = etendre_oracle;

### Exercice 3 — projeter la pente du DPLL

La section 1c mesure les noeuds DPLL moyens pour $n$ croissant et en déduit un facteur de croissance
par variable ajoutée. Projeter le temps de résolution attendu à $n = 40$ puis $n = 60$, et estimer
le plus petit $n$ où la projection dépasse une heure.

In [12]:
// Exercice 3 : a completer
// facteur_temps n'existe plus a cette cellule (autonomie evcxr) : le recalculer
// (ou recopier sa valeur) depuis la derniere ligne du tableau 1c (n = 30).
let temps_projete_40: Option<f64> = None; // TODO etudiant : projection du temps a n = 40 (secondes)
let n_une_heure: Option<i64> = None; // TODO etudiant : plus petit n ou la projection depasse 3600 s
// Indice : temps(n) ~ temps(30) * facteur_temps ** (n - 30) ; le logarithme retourne n_une_heure.
let _exercice_3 = (temps_projete_40, n_une_heure);

## 6. Conclusion

| Geste | Où | Ce qui a été mesuré |
|---|---|---|
| Tourner les inclusions | §1 | Atteignabilité décidée deux fois : vérificateur à 2 compteurs contre pile DFS ($L \subseteq NL$ visible) ; 2-SAT linéaire ($n = 800$ en milliseconde) contre 3-SAT DPLL à croissance multiplicative par variable (facteur mesuré), certificats vérifiés 20/20 en polynomial |
| Construire les deux mondes | §2 | Oracle diagonal : 4 machines vaincues, témoin exhaustif d'accord (BGS monde A exécuté) ; oracle QBF : la machine $P$ égale $NP$ en une requête (monde B) |
| Cartographier honnêtement | §3 | 8 arêtes : 3 à témoin exécuté, 2 citées avec leur origine (hiérarchie 03/03b), 3 ouvertes, chacune avec sa raison propre (relativisation, certificateur général, arithmétisation) |
| Dater le gap | §4 | Mathlib : calculabilité oui, Zoo non — aucune classe à bornage, aucun oracle ; grade écrit |

Le fil du carnet : rendre **visibles** les inclusions en les exécutant, rendre **audible** la question ouverte en construisant deux mondes qui y répondent différemment, puis dire honnêtement — carte à l'appui — ce qui reste hors de portée de l'exécution.


## À retenir

1. **Deux procédures illustrent recherche et vérification.** Le vérificateur à 2 mots de mémoire et la pile DFS s'accordent sur toute la famille ($n$ = 40, 100, 250, 600 : `accord=True`), et le faux témoin — une arrivée amputée — est refusé : le vérificateur n'est donc pas une tautologie.
2. **La pile DFS paie son espace.** De 16 cases explorées à $n = 40$ à **190** à $n = 600$, pendant que les temps sont affichés dans la cellule de mesure ; le vérificateur NL, lui, rend son verdict en 4 à 9 pas selon l'instance, avec **2** mots de mémoire.
3. **3-SAT pousse, 2-SAT ne pousse pas.** Le DPLL (densité 4,6$n$) passe de 10,3 nœuds à $n = 10$ à 63,9 nœuds à $n = 30$, soit un facteur mesuré de **≈1,10×** par variable ajoutée (≈1,20× en temps) ; le 2-SAT de Kosaraju suit un algorithme linéaire (temps affiché pour $n = 800$). Les 20 certificats 3-SAT sont vérifiés en temps polynomial : **20/20**.
4. **Deux mécanismes finis associés à Baker–Gill–Solovay sont illustrés.** Monde A : les 4 machines P sont vaincues à chaque stade, et le témoin exhaustif côté NP confirme $L_O$ ; monde B : la machine P répond en **1 requête**, exactement comme la NP exhaustive.
5. **La carte est honnête sur sa couverture.** Huit arêtes : **3 illustrées sur des exemples finis dans ce carnet**, 2 citées avec leur origine, 3 ouvertes — dont P vs NP, que les deux mondes d'oracles ne tranchent pas.

## Vérifiez votre compréhension

1. Le carnet mesure l'accord entre le vérificateur NL et la pile DFS sur toute la famille, puis montre qu'un faux témoin est refusé. Quel est le rôle exact de ces deux contrôles dans l'argument « $L \subseteq NL$ » ?
2. Entre 2-SAT et 3-SAT, le carnet compare un facteur de croissance par variable ajoutée (≈1,10× en nœuds DPLL) plutôt que les temps. Pourquoi comparer les temps serait-il trompeur ?
3. Dans le monde A, la machine P est vaincue à chacun des quatre stades, et le témoin exhaustif côté NP confirme $L_O$. Que prouve ce double accord, et que ne prouve-t-il pas ?
4. Sur les huit arêtes de la carte, trois seulement ont un témoin exécuté dans ce carnet. Quel est le risque, pour un lecteur, si les arêtes ouvertes étaient présentées comme mesurées ?

<details><summary>Réponses</summary>

1. Le premier contrôle prouve la **cohérence** (deux méthodes indépendantes disent la même chose) ; le second prouve que le vérificateur n'accepte pas tout (il refuse un mauvais certificat). Sans le second, un vérificateur qui accepte tout donnerait un accord parfait.
2. Les temps dépendent de la taille absolue des instances : opposer du 2-SAT à $n = 800$ et du 3-SAT à $n = 30$ compare deux tailles et deux implémentations. Le facteur par variable ajoutée normalise la taille et isole la croissance.
3. Il prouve que, dans cet univers d'oracles, la diagonale fonctionne comme [03b](Complexity-03b-HartmanisStearns-HierarchieTemporelle-Python.ipynb) l'a établi : chaque machine P se fait contrarier sur une entrée, et l'on ne s'écarte jamais de la vérité. Il ne prouve rien de P contre NP **dans l'arithmétique ordinaire** — c'est précisément ce que le monde B (où $P = NP$) interdit à toute preuve par diagonalisation.
4. Un lecteur prendrait une frontière non mesurée pour un résultat établi. La distinction Mesuré / Cité / Ouvert est ce qui rend la carte utilisable : elle dit où l'on peut vérifier de soi-même et où l'on ne peut que lire une source.

</details>

## Pour aller plus loin

- **Carnet suivant** : [Complexity-03d — la réfutation annoncée de l'ETH](Complexity-03d-ETH-Refutation-Annoncee-Lean-Python.ipynb) — une annonce de réfutation, lue à la source et confrontée au mur mesuré.
- **Dans la même série** : [Complexity-04 — Décider sans connaître la suite](Complexity-04-OnlineAlgorithms-Python.ipynb) — l'algorithme en ligne, l'autre moitié du Zoo ; [Complexity-05 — Compter est plus dur que vérifier](Complexity-05-CountingHarder-Permanent-Python.ipynb) — la classe #P et la permanent.
- **Carnet voisin d'une autre série** : [CSP-3 — Advanced](../../Search/Part2-CSP/CSP-03-Advanced-Python.ipynb) — le solveur SAT et ses budgets, dans une autre série.
- **Lectures** : Baker, Gill et Solovay (1975), *Relativizations of the P=?NP Question* (copie archivée au gisement partagé, dossier `Bibliographie IA/Automata/`) ; *The CP-SAT Primer — Using and Understanding Google OR-Tools' CP-SAT Solver* (2023), pour le solveur SAT de référence et la lecture de ses coûts (dossier `Bibliographie IA/Constraint Programming/`).